In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torchmetrics

import numpy as np
import os
import glob
import re
from tqdm import tqdm
from omegaconf import OmegaConf

import matplotlib.pyplot as plt

from lightning.pytorch import seed_everything

from snpgen.utils import instantiate_from_config
from snpgen.data.loader import SplitDataset

# Inference module for VAE reconstructions
from snpgen.inference import (
    ReconstructionGenerator,
    save_synthetic_dataset,
    get_output_filename,
    dataset_exists,
)

OmegaConf.register_new_resolver("eval", eval)

## User Settings

In [ ]:

# ============================================================
# USER SETTINGS - Configure these before running the notebook
# ============================================================

# Path to the VAE checkpoint to load
reload_path = '/path/to/snpgen/checkpoints/trait1/trait1_vae_disc_emb128_small_white-31532396/epoch=398-step=514710-val_accuracy_recons=0.921.ckpt'

# Which splits to generate reconstructions for
# Options: 'train', 'val', 'test', 'train_val', 'full' (or list of multiple splits)
splits_to_reconstruct = ['train_val', 'test']

# Whether to sample from the posterior (stochastic) or use mean (deterministic)
sample_posterior = True

# Whether to store latent space info (mu, logvar, z) - increases memory usage
store_latents = False

# Whether to store original samples in the output file (for analysis)
store_originals = False

# Sampling batch size for reconstruction generation
batch_size = 256 * 3

In [ ]:
seed = 42
seed_everything(seed, workers=True)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") 

In [ ]:
NUM_WORKERS = int(os.environ["SLURM_CPUS_PER_TASK"])
NUM_NODES = int(os.environ["SLURM_NNODES"])
ALLOCATED_GPUS_PER_NODE = int(os.environ["SLURM_GPUS_ON_NODE"])
SLURM_JOBID = os.environ["SLURM_JOB_ID"]

In [ ]:
num_gpus = torch.cuda.device_count()
print(f"{num_gpus} GPU(s) available")
print(f"Using {NUM_WORKERS} workers for the DataLoader")

# Load Config

In [ ]:
config_path = os.path.join(os.path.dirname(reload_path), 'config.yaml')
assert os.path.exists(config_path), f"config.yaml not found at {config_path}. This notebook requires a run with saved config.yaml."
config = OmegaConf.load(config_path)

In [ ]:
resolved_config_dict = OmegaConf.to_container(config, resolve=True)
config_orig = config.copy() # keep a backup of the original config prior to any change

# Build Dataset

In [ ]:
if 'dataset_path' in config:
    h5_path = config['dataset_path']
else:
    raise ValueError("dataset_path not found in config. Please ensure the config.yaml contains the dataset_path key pointing to the original dataset used for training.")
    
print(h5_path)

In [ ]:
print(f"Loading Dataset from: {h5_path}")
if config.get('data', {}).get('raw_dataset', None):
    raw_dataset = instantiate_from_config(config.data.raw_dataset, file_path=h5_path, metadata=True, seed=seed)
else:
    raise ValueError("raw_dataset config not found. Please ensure the config.yaml contains a data.raw_dataset section with the appropriate dataset configuration.")

In [ ]:
if config.get('data', {}).get('dataset', None):
    train_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('train'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)
    val_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('val'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)
    test_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('test'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)
    
    train_val_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('train_val'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)
    complete_dataset = instantiate_from_config(config.data.dataset, raw_dataset.get_split('full'), block_ids=raw_dataset.get_metadata('full', 'block_id') if hasattr(raw_dataset, 'get_metadata') else None)

else:
    raise ValueError("dataset config not found. Please ensure the config.yaml contains a data.dataset section with the appropriate dataset configuration.")

# Setup DataLoaders

In [ ]:
train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
    #sampler=ImbalancedDatasetSampler(train_dataset, strategy='inverse_freq'), # balance dataset on labels (which also implicitly performs shuffling)
)

val_dataloader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

train_val_dataloader = torch.utils.data.DataLoader(
    train_val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

complete_dataloader = torch.utils.data.DataLoader(
    complete_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

# Build Models

In [ ]:
config.model.params.autoencoder_config.params.ckpt_path = reload_path
config.model.params.autoencoder_config.params.load_ema_ckpt = True

if 'seq_len' in config:
    config.seq_len = train_dataset.get_seq_len()

In [ ]:
vae_training_wrapper = instantiate_from_config(config.model)

# Generate and Save VAE Reconstructions

This section generates reconstructions through the VAE encoder-decoder pipeline.
The reconstructions are saved in HDF5 format compatible with the ML/PRS training pipeline,
allowing us to evaluate whether the VAE preserves the predictive information in the data.

**Note:** The VAE is not conditioned on phenotype labels, so we save the real targets
alongside the reconstructions. This allows fair comparison with the original data
when training downstream ML/PRS models.

In [ ]:
# Output directory (same as VAE checkpoint directory)
output_dir = os.path.dirname(reload_path)

# =============================================================================
# GENERATE RECONSTRUCTIONS
# =============================================================================

# Create the reconstruction generator
recon_generator = ReconstructionGenerator(vae_training_wrapper, device='cuda')
recon_generator.prepare_model()

# Map split names to dataloaders
split_dataloaders = {
    'train': train_dataloader,
    'val': val_dataloader,
    'test': test_dataloader,
    'train_val': train_val_dataloader,
    'full': complete_dataloader,
}

# Check if data is one-hot encoded
onehot = config.data.raw_dataset.params.get('onehot', True)

for split in splits_to_reconstruct:
    print(f"\n{'='*60}")
    print(f"Processing split: {split}")
    print(f"{'='*60}")
    
    # Get output filename
    output_filename = get_output_filename(
        base_name='vae',
        mode='reconstruction',
        split=split
    )
    output_path = os.path.join(output_dir, output_filename)
    
    # Check if already exists
    if dataset_exists(output_dir, output_filename):
        print(f"Reconstruction dataset already exists: {output_path}")
        print("Skipping generation...")
        continue
    
    # Get the appropriate dataloader
    if split not in split_dataloaders:
        print(f"Warning: Unknown split '{split}', skipping...")
        continue
    
    dataloader = split_dataloaders[split]
    
    # Get pad mask if available
    pad_mask = getattr(dataloader.dataset, 'pad_mask', None)
    
    # Generate reconstructions using the inference module
    result = recon_generator.generate_reconstructions(
        dataloader,
        sample_posterior=sample_posterior,
        store_latents=store_latents,
        store_originals=store_originals,
        onehot=onehot,
        pad_mask=pad_mask,
        verbose=True
    )
    
    # ==========================================================================
    # EXTRACT EIDS FROM METADATA
    # ==========================================================================
    # Since we use shuffle=False in dataloaders, the order is preserved and
    # matches the split indices from raw_dataset.
    
    # Get metadata for this split
    _, original_targets, split_metadata = raw_dataset.get_split(split, metadata=True)
    
    if split_metadata is not None:
        # Check that dataloader targets match original targets otherwise metadata alignment is off
        assert np.all(original_targets == result.targets), "Mismatch between original targets and dataloader targets! Are dataloaders using shuffle=False?"
        
        result.eids = split_metadata.get('eids', None)
    
    # Save the reconstruction dataset
    # Uses 'syn_samples' key for compatibility with SplitDataset loader
    save_synthetic_dataset(
        result=result,
        output_path=output_path,
        mode='reconstruction',
        extra_attrs={
            'sample_posterior': sample_posterior,
            'vae_checkpoint': reload_path,
            'split': split,
        }
    )
    
    print(f"\nSaved reconstruction dataset to: {output_path}")
    print(f"  - Reconstructions shape: {result.reconstructions.shape}")
    print(f"  - Targets shape: {result.targets.shape}")
    if result.orig_samples is not None:
        print(f"  - Original samples shape: {result.orig_samples.shape}")
    if result.eids is not None:
        print(f"  - EIDs shape: {result.eids.shape}")

# Keep results from last split for analysis
reconstructions = result.reconstructions
targets = result.targets
orig_samples = result.orig_samples
eids = result.eids

# Create torch tensors for analysis
reconstructions_torch = torch.from_numpy(reconstructions)
orig_samples_torch = torch.from_numpy(orig_samples) if orig_samples is not None else None
targets_torch = torch.from_numpy(targets)

# Transpose for SNP-wise analysis
orig_samples_T = np.transpose(orig_samples) if orig_samples is not None else None
reconstructions_T = np.transpose(reconstructions)